# 15. Integrált Ingatlan Kereső és Döntéstámogató Vezérlőpult

**Cél:** a korábbi fejezetek adatbázisának és prediktív modelljeinek szintetizálása egyetlen, végfelhasználó-központú, interaktív dashboarddá.

---

### 📖 Az elmélettől a gyakorlatig
A feltárt térökonometriai összefüggések, a környezeti terhelések (vasúti diszkont), a szolgáltatási hálózatok és a Random Forest-predikciók itt egyetlen dinamikus szűrőrendszerben találkoznak. A dashboard a **data-driven real estate** szemléletet bizonyítja: a befektetői döntések többdimenziós prediktív analitikára támaszkodnak.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
display(HTML(render_area_header_html(df=df)))
elado = df[df['listing_type'] == 'elado'].copy()
pontos = elado[elado['minta_garantalt_pontos'] == 1].copy()

# 1. Kiemelt Kutatási Főmutatók — a MODELJEK FUTTATOTT EREDMÉNYEIBŐL (nem literálok)
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score

# (a) Kompakt hedonikus modell (a 07-es specifikáció) — minden KPI innen
mdf = pontos.dropna(subset=['log_nm_ar', 'korrigalt_alapterulet_nm', 'szobaszam_osszes', 'is_panel',
                            'has_lift', 'has_erkely', 'allapot_kod', 'emelet_szam', 'epulet_kora_ev',
                            'tavolsag_metro_halozati_m', 'tavolsag_vasut_m', 'tavolsag_vasut_halozati_m']).copy()
# KÜSZÖB-ALAPÚ főmodell (a kánon szerint): TOD = diszjunkt izokrón dummyk,
# immisszió = kanonikus sáv-dummyk; a folytonos alakok csak robusztusságvizsgálatra.
_f = ['korrigalt_alapterulet_nm', 'szobaszam_osszes', 'is_panel', 'has_lift', 'has_erkely',
      'allapot_kod', 'emelet_szam', 'epulet_kora_ev']
# KANONIKUS specifikáció (a 04/07/15 közös modellje — azonos számok mindenhol)
_canon = fit_canonical_hedonic(mdf)
_hmod = _canon['model']
_hmod_z = _canon['model_zones']  # zaj-fókuszú (a zajdiszkont fejléce)
_f = _canon['phys']
_tod = _canon['tod_cols']
_kp = _canon['kp_cols']
_z_d = _canon['X'][_canon['zone_cols']]
_z_ref = _canon['ref_zone']
print(f"Kanonikus főmodell: N={int(_hmod.nobs)}, R²={_hmod.rsquared:.3f} | vasúti TOD: {_tod} | "
      f"kötöttpálya: {_kp} | immissziós referencia: {_z_ref}")
def _pct(col, mod=None):
    mod = mod if mod is not None else _hmod
    if col not in mod.params.index:
        return (np.nan, np.nan, np.nan, 1.0)
    b = mod.params[col]; p = mod.pvalues[col]
    lo = (np.exp(_hmod.conf_int().loc[col, 0]) - 1) * 100
    hi = (np.exp(_hmod.conf_int().loc[col, 1]) - 1) * 100
    return ((np.exp(b) - 1) * 100, lo, hi, p)
panel_pct, panel_lo, panel_hi, panel_p = _pct('is_panel')
_zaj_col = '<150 m' if '<150 m' in _z_d.columns else _z_d.columns[0]
zaj_pct, zaj_lo, zaj_hi, zaj_p = _pct(_zaj_col, mod=_hmod_z)
_tod_col = 'tod_0_375' if 'tod_0_375' in _tod else _tod[0]
tod_pct, tod_lo, tod_hi, tod_p = _pct(_tod_col)
allapot_pct, allapot_lo, allapot_hi, allapot_p = _pct('allapot_kod')
erkely_pct, erkely_lo, erkely_hi, erkely_p = _pct('has_erkely')
lift_pct, lift_lo, lift_hi, lift_p = _pct('has_lift')
# (a metró nem szerepel külön a főmodellben: a kötöttpálya-index dummyi tartalmazzák)

# (b) SAR ρ (kompakt GM_Lag)
try:
    from libpysal.weights import KNN
    from spreg import GM_Lag
    _mid = float(mdf['geokodolt_lat'].median())
    _xy = np.column_stack((mdf['geokodolt_lon'] * 111320.0 * np.cos(np.radians(_mid)),
                           mdf['geokodolt_lat'] * 110540.0))
    _w = KNN.from_array(_xy, k=8); _w.transform = 'R'
    _f_sar = _f + _tod + _kp
    _gm = GM_Lag(mdf['log_nm_ar'].values.reshape(-1, 1), mdf[_f_sar].values, w=_w,
                 name_y='log_nm_ar', name_x=_f_sar)
    rho_val = float(_gm.betas[-1][0]); mult_val = 1.0 / (1.0 - rho_val) if rho_val < 1 else np.nan
except Exception as ex:
    print(f"SAR nem futott a dashboardban ({ex}) — a multiplikátor nem jelenik meg.")
    rho_val = mult_val = np.nan

# (c) Rent gap (kompakt, a 12-es párosított módszerrel)
kiado = df[df['listing_type'] == 'kiado'].copy()
def _fit(sub, ycol):
    s = sub.dropna(subset=[ycol, 'korrigalt_alapterulet_nm', 'szobaszam_osszes', 'allapot_kod',
                           'is_panel', 'has_lift', 'has_erkely']).copy()
    X = sm.add_constant(s[['korrigalt_alapterulet_nm', 'szobaszam_osszes', 'allapot_kod',
                           'is_panel', 'has_lift', 'has_erkely']])
    return sm.OLS(s[ycol], drop_constant_columns(X)).fit()
try:
    m_el = _fit(elado, 'log_nm_ar'); m_ki = _fit(kiado, 'log_ar')
    ref = pd.DataFrame([{'korrigalt_alapterulet_nm': 50, 'szobaszam_osszes': 2,
                         'allapot_kod': 3, 'is_panel': 0, 'has_lift': 1, 'has_erkely': 1}])
    _order = ['const', 'korrigalt_alapterulet_nm', 'szobaszam_osszes', 'allapot_kod',
              'is_panel', 'has_lift', 'has_erkely']
    Xr = pd.DataFrame(np.array([[1.0, 50, 2, 3, 0, 1, 1]]), columns=_order)
    p_el = float(np.exp(m_el.predict(Xr[m_el.model.exog_names])[0]))
    p_ki = float(np.exp(m_ki.predict(Xr[m_ki.model.exog_names])[0]))
    cap = 0.05
    rent_gap = (p_ki * 12 / cap) - (p_el * 50)
except Exception as ex:
    print(f"Rent gap nem számítható ({ex})")
    rent_gap = np.nan

# (d) RF R² térbeli CV-vel
try:
    _feats = [c for c in ['korrigalt_alapterulet_nm', 'szobaszam_osszes', 'is_panel', 'has_lift',
                          'has_erkely', 'allapot_kod', 'emelet_szam', 'epulet_kora_ev',
                          'kotottpalya_15p_seta', 'tavolsag_vasut_m', 'vasut_15p_seta',
                          'is_vasut_immisszio_150m'] if c in mdf.columns]
    _ml = mdf.dropna(subset=['nm_ar_huf'] + _feats)
    _blk = KMeans(n_clusters=8, random_state=42, n_init=10).fit_predict(
        np.column_stack((_ml['geokodolt_lon'] * 111320.0 * np.cos(np.radians(_mid)),
                         _ml['geokodolt_lat'] * 110540.0)))
    _pred = cross_val_predict(RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
                              _ml[_feats], _ml['nm_ar_huf'], cv=GroupKFold(n_splits=8), groups=_blk)
    rf_r2 = r2_score(_ml['nm_ar_huf'], _pred)
except Exception as ex:
    print(f"RF nem futott ({ex})")
    rf_r2 = np.nan

def _fmt_pct(v):
    return '—' if not np.isfinite(v) else f"{v:+.1f}%"

kpi_master = [
    ("Panel Diszkont", _fmt_pct(panel_pct), f"HC1 p={panel_p:.3f}", "#dc2626"),
    ("Vasúti Zajdiszkont", _fmt_pct(zaj_pct), f"{_zaj_col} sáv vs. {_z_ref} (HC1 p={zaj_p:.3f})", "#ef4444"),
    ("Vasútállomás TOD", _fmt_pct(tod_pct), f"{_tod_col} (≤375 m séta) vs. >1125 m — HC1 p={tod_p:.3f}", "#10b981"),
    ("Térbeli Multiplikátor", (f"{mult_val:.2f}x" if np.isfinite(mult_val) else '—'), f"SAR ρ = {rho_val:.3f}" if np.isfinite(rho_val) else '—', "#2563eb"),
    ("Referencia Rent Gap (50 m²)", (fmt_mft(rent_gap / 1e6) if np.isfinite(rent_gap) else '—'), "Párosított becslés, 5% cap", "#7c3aed"),
    ("Random Forest R² (OOS térbeli CV)", (f"{rf_r2:.3f}" if np.isfinite(rf_r2) else '—'), "Szivárgásmentes", "#059669"),
    ("Állapot Prémium / kategória", _fmt_pct(allapot_pct), f"HC1 p={allapot_p:.3f}", "#0891b2")
]
display(HTML(kpi_grid_html(kpi_master)))


### 1. Hatásmátrix: Az Ingatlanárakat Meghatározó Tényezők (Forest Plot)

Az alábbi ábra összefoglalja az összes azonosított fizikai, környezeti és térbeli tényező tiszta gazdasági hatását és 95%-os megbízhatósági intervallumát.

In [ ]:
# 2. Összesített Hatásmátrix — a BECSÜLT küszöb-alapú hedonikus modellből (HC1 CI-k)
hatasok = pd.DataFrame([
    {'Tenyezo': 'Panelszerkezet', 'Hatas_pct': panel_pct, 'CI_low': panel_lo, 'CI_high': panel_hi, 'Kategoria': 'Fizikai'},
    {'Tenyezo': f'Vasúti zaj ({_zaj_col} sáv)', 'Hatas_pct': zaj_pct, 'CI_low': zaj_lo, 'CI_high': zaj_hi, 'Kategoria': 'Környezeti'},
    {'Tenyezo': 'Állapotfelár (kategóriánként)', 'Hatas_pct': allapot_pct, 'CI_low': allapot_lo, 'CI_high': allapot_hi, 'Kategoria': 'Fizikai'},
    {'Tenyezo': 'Erkély megléte', 'Hatas_pct': erkely_pct, 'CI_low': erkely_lo, 'CI_high': erkely_hi, 'Kategoria': 'Fizikai'},
    {'Tenyezo': 'Lift megléte', 'Hatas_pct': lift_pct, 'CI_low': lift_lo, 'CI_high': lift_hi, 'Kategoria': 'Fizikai'},
    {'Tenyezo': f'Vasútállomás TOD ({_tod_col}, ≤375 m)', 'Hatas_pct': tod_pct, 'CI_low': tod_lo, 'CI_high': tod_hi, 'Kategoria': 'Közlekedés'},
]).sort_values('Hatas_pct', ascending=True)

fig1 = go.Figure()
for kat, col in [('Fizikai', '#2563eb'), ('Környezeti', '#dc2626'), ('Közlekedés', '#059669')]:
    sub = hatasok[hatasok['Kategoria'] == kat]
    fig1.add_trace(go.Scatter(
        x=sub['Hatas_pct'], y=sub['Tenyezo'], mode='markers', marker=dict(size=12, color=col),
        error_x=dict(type='data', symmetric=False,
                     array=sub['CI_high'] - sub['Hatas_pct'],
                     arrayminus=sub['Hatas_pct'] - sub['CI_low'],
                     color=col, thickness=2, width=6),
        name=kat
    ))
fig1.add_vline(x=0, line_dash='dash', line_color='black', opacity=0.7)
fig1.update_layout(
    title=f'Hatásmátrix: implicit árhatások és 95%-os HC1 konfidencia-intervallumok (%) — referencia: {_z_ref} / >1125 m',
    xaxis_title='Várható hatás a fajlagos ingatlanárra (%)',
    template=PLOTLY_TEMPLATE, height=480,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig1.show()


### 2. A Vasút Kettős Természete: Immisszió vs. Állomási Elérhetőség

A vasút jelenléte a vizsgált területen egyszerre jelent negatív externáliát (zaj, rezgés) és pozitív externáliát (gyors kötöttpályás elérhetőség). Az alábbi szintézis-diagram bemutatja e két ellentétes erő eredőjét.

In [ ]:
# 3. Kettős hatásgörbe — a BECSÜLT sávdiszkontokból és TOD-együtthatókból
sav_mid = {'<150 m': 75, '150-300 m': 225, '300-500 m': 400, '500-1000 m': 750, '1000-2000 m': 1500, '>2000 m (referencia)': 2200}
zaj_pts, zaj_x = [], []
for z in _z_d.columns:
    if z in _hmod.params:
        zaj_pts.append((np.exp(_hmod.params[z]) - 1) * 100)
        zaj_x.append(sav_mid.get(z, 1000))
x_tav = np.linspace(50, 2000, 200)
from scipy.optimize import curve_fit
def _decay(x, a, b):
    return a * np.exp(-x / b)
if len(zaj_x) >= 2 and np.nanmax(np.abs(np.array(zaj_pts))) > 0:
    try:
        popt, _ = curve_fit(_decay, np.array(zaj_x), np.array(zaj_pts), p0=(-20, 300), maxfev=5000)
        zaj_gorbe = _decay(x_tav, *popt)
    except Exception:
        zaj_gorbe = np.interp(x_tav, np.array(zaj_x), np.array(zaj_pts))
else:
    zaj_gorbe = np.zeros_like(x_tav)
# TOD görbe: a becsült 5p-prémium lecsengése az állomás körül
tod_gorbe = tod_pct * np.exp(-((x_tav) ** 2) / (2 * 375 ** 2))
eredo_gorbe = zaj_gorbe + tod_gorbe

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=x_tav, y=zaj_gorbe, mode='lines', line=dict(color='#dc2626', width=2, dash='dash'), name='Zajdiszkont (illesztett lecsengés)'))
fig2.add_trace(go.Scatter(x=zaj_x, y=zaj_pts, mode='markers', marker=dict(color='#dc2626', size=9, symbol='x'), name='Becsült sávdiszkontok'))
fig2.add_trace(go.Scatter(x=x_tav, y=tod_gorbe, mode='lines', line=dict(color='#059669', width=2, dash='dot'), name='TOD prémium (becslésből)'))
fig2.add_trace(go.Scatter(x=x_tav, y=eredo_gorbe, mode='lines', line=dict(color='#2563eb', width=4), name='Nettó eredő hatás'))
fig2.add_hline(y=0, line_dash='solid', line_color='black', opacity=0.3)
fig2.add_vline(x=150, line_dash='dash', line_color='#dc2626', annotation_text='150m immissziós határ')
fig2.add_vline(x=750, line_dash='dash', line_color='#059669', annotation_text='750m sétahatár')
fig2.update_layout(
    title='A vasút kettős gazdasági hatásgörbéje (becsült együtthatókból)',
    xaxis_title='Távolság a vasúti infrastruktúrától (méter)',
    yaxis_title='Becsült tiszta árhatás (%)',
    template=PLOTLY_TEMPLATE, height=450
)
fig2.show()


### 3. Városrészi Összehasonlító Radar Chart és Rendszerszintű Profil

A városrészek többdimenziós lakáspiaci és infrastrukturális összehasonlítása normalized (0-100) skálán.

In [ ]:
# 4. Városrészi Radar Profil
radar_metrics = elado.groupby('varosresz', observed=True).agg({
    'nm_ar_huf': 'median',
    'is_panel': 'mean',
    'allapot_kod': 'mean',
    'alapterulet_nm': 'median',
    'tavolsag_metro_halozati_m': 'median'
}).reset_index()

# Normalizálás 0-100 skálára
categories = ['Fajlagos Ár', 'Panel Arány', 'Műszaki Állapot', 'Átlagos Méret', 'Metró Közeliség']
fig3 = go.Figure()

def _mm(x):
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo) * 100 if hi > lo else np.full_like(x, 50.0)
radar_metrics['_mm_ar'] = _mm(radar_metrics['nm_ar_huf'])
radar_metrics['_mm_panel'] = _mm(radar_metrics['is_panel'])
radar_metrics['_mm_allapot'] = _mm(radar_metrics['allapot_kod'])
radar_metrics['_mm_meret'] = _mm(radar_metrics['alapterulet_nm'])
radar_metrics['_mm_metro'] = _mm(radar_metrics['tavolsag_metro_halozati_m'].max() - radar_metrics['tavolsag_metro_halozati_m'])
for _, row in radar_metrics.iterrows():
    vals = [
        row['_mm_ar'], row['_mm_panel'], row['_mm_allapot'], row['_mm_meret'], row['_mm_metro']
    ]
    vals.append(vals[0]) # lezárás
    
    fig3.add_trace(go.Scatterpolar(
        r=vals,
        theta=categories + [categories[0]],
        fill='toself',
        name=row['varosresz']
    ))

fig3.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title=f"Városrészi lakáspiaci profil — {os.environ.get('TDK_ACTIVE_AREA', 'kobanya')} (radar, min-max normalizálva)",
    template=PLOTLY_TEMPLATE,
    height=520
)
fig3.show()

In [ ]:
# 4. Objektív, dinamikus döntéstámogató összefoglaló — minden szám a fenti becsült modellekből
from IPython.display import display, Markdown

def _irany(v):
    if not np.isfinite(v):
        return '—'
    return 'diszkont' if v < 0 else 'prémium'

def _fmt_p(v):
    return f"p = {v:.3f}" if np.isfinite(v) else '—'

_summary_md = f"""### 4. A Becsült Főhatások Objektív Összefoglalója

A vezérlőpult minden mutatója a korábbi cellákban **futtatott modellek** becsült eredményeiből származik (kanonikus hedonikus modell, SAR, párosított rent-gap becslés és out-of-sample Random Forest), ezért minden aktív mintán újraszámolódik.

| Becsült hatás | Irány | Becsült mérték | 95% CI (HC1) | Statisztika |
| :--- | :--- | ---: | :--- | :--- |
| Panelszerkezet | {_irany(panel_pct)} | {_fmt_pct(panel_pct)} | {_fmt_pct(panel_lo)} – {_fmt_pct(panel_hi)} | {_fmt_p(panel_p)} |
| Vasúti zaj ({_zaj_col} vs. {_z_ref}) | {_irany(zaj_pct)} | {_fmt_pct(zaj_pct)} | {_fmt_pct(zaj_lo)} – {_fmt_pct(zaj_hi)} | {_fmt_p(zaj_p)} |
| Állomáskörnyéki elérhetőség ({_tod_col}, ≤375 m séta) | {_irany(tod_pct)} | {_fmt_pct(tod_pct)} | {_fmt_pct(tod_lo)} – {_fmt_pct(tod_hi)} | {_fmt_p(tod_p)} |
| Műszaki állapot (kategóriánként) | {_irany(allapot_pct)} | {_fmt_pct(allapot_pct)} | {_fmt_pct(allapot_lo)} – {_fmt_pct(allapot_hi)} | {_fmt_p(allapot_p)} |

**Térbeli függés és prediktív robusztusság:**

| Mutató | Érték | Értelmezés |
| :--- | :--- | :--- |
| Térbeli autokorreláció (SAR ρ) | {(f'{rho_val:.3f}' if np.isfinite(rho_val) else '—')} | a hibatagok térbeli függésének erőssége a főmodellben |
| Térbeli multiplikátor 1/(1−ρ) | {(f'{mult_val:.2f}' if np.isfinite(mult_val) else '—')} | egy lokális sokk térbeli tovagyűrűzésének szorzója |
| Random Forest R² (out-of-sample, térbeli CV) | {(f'{rf_r2:.3f}' if np.isfinite(rf_r2) else '—')} | a gépi tanulásos előrejelzés pontossága az aktív mintán |
| Referencia Rent Gap (50 m²) | {fmt_mft(rent_gap / 1e6)} | becsült értékrés a referencia-lakás bérleti és tulajdoni értéke között |

**💡 Objektív olvasat:** a fenti együtthatók az aktív minta saját becslései, nem előre rögzített értékek — a dashboard ezekből építi fel a hatásmátrixot, a kettős hatásgörbét és az arbitrázs-keresőt.
"""
display(Markdown(_summary_md))

In [ ]:
# 6. Diagram- és térképgaléria: minden típus × változó-választó (JS)
# A galéria minden fő diagramtípust tartalmaz; a JS-választók a statikus
# HTML-exportban is működnek (a Plotly-ábrák szerveroldalon renderelődnek,
# a választók CSS-megjelenítést kapcsolnak).
gal_vars = {
    'nm_ar_huf': 'Négyzetméterár (HUF/m²)',
    'log_nm_ar': 'Log fajlagos ár',
    'korrigalt_alapterulet_nm': 'Korrigált alapterület (m²)',
    'szobaszam_osszes': 'Szobaszám',
    'allapot_kod': 'Állapot index (1-6)',
    'epulet_kora_ev': 'Épületkor (év)',
    'tavolsag_vasut_m': 'Vágánytávolság (m)',
    'tavolsag_vasut_halozati_m': 'Állomás hálózati táv (m)',
    'tavolsag_kotottpalya_halozati_m': 'Kötöttpálya táv (m)',
    'tavolsag_metro_halozati_m': 'Metró táv (m)',
}
_num = [c for c in gal_vars if c in mdf.columns and mdf[c].notna().sum() > 10]

figs = {}
for v in _num:
    gv = gal_vars[v]
    _d = mdf.dropna(subset=[v])
    figs[v] = {
        'scatter': px.scatter(_d, x='korrigalt_alapterulet_nm', y='nm_ar_huf', color=v,
                              title=f'Alapterület–ár, szín: {gv}', template=PLOTLY_TEMPLATE,
                              marginal_x='histogram'),
        'box': px.box(_d, x='vasut_zona', y=v, color='vasut_zona',
                      title=f'{gv} eloszlása immissziós sávonként', template=PLOTLY_TEMPLATE),
        'violin': px.violin(_d, x='vasut_zona', y=v, box=True,
                            title=f'{gv} — hegedűdiagram sávonként', template=PLOTLY_TEMPLATE),
        'hist': px.histogram(_d, x=v, nbins=40, title=f'{gv} hisztogram', template=PLOTLY_TEMPLATE),
        'map_scatter': px.scatter_map(_d, lat='geokodolt_lat', lon='geokodolt_lon', color=v,
                                         size='korrigalt_alapterulet_nm', size_max=12,
                                         map_style='carto-positron', zoom=11,
                                         title=f'{gv} térképen (scatter)', height=430),
        'map_density': px.density_map(_d, lat='geokodolt_lat', lon='geokodolt_lon', z=v,
                                         radius=18, map_style='carto-positron', zoom=11,
                                         title=f'{gv} sűrűségtérkép', height=430),
    }

_gal_html = '<div style="background:#f8fafc;border:1px solid #cbd5e1;border-radius:12px;padding:16px;margin:16px 0;">'
_gal_html += '<h3 style="margin:0 0 10px 0;">Diagram- és térképgaléria</h3>'
_gal_html += ('<div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:12px;">'
              '<label>Változó: <select id="gal_var" style="padding:6px;">')
for c in _num:
    _gal_html += f'<option value="{c}">{gal_vars[c]}</option>'
_gal_html += '</select></label><label>Típus: <select id="gal_typ" style="padding:6px;">'
for t, tn in [('scatter', 'Szórás'), ('box', 'Doboz'), ('violin', 'Hegedű'),
              ('hist', 'Hisztogram'), ('map_scatter', 'Térkép pont'), ('map_density', 'Térkép sűrűség')]:
    _gal_html += f'<option value="{t}">{tn}</option>'
_gal_html += '</select></label></div>'
_gal_html += '<div id="gal_grid" style="display:grid;gap:12px;grid-template-columns:repeat(auto-fit,minmax(420px,1fr));">'
for v in _num:
    for t, fig in figs[v].items():
        _gal_html += (f'<div class="gal_cell" data-var="{v}" data-typ="{t}" '
                      f'style="background:#fff;border-radius:8px;padding:6px;">{fig.to_html(full_html=False, include_plotlyjs=False)}</div>')
_gal_html += '</div>'
_gal_html += '<script>'
_gal_html += '(function(){'
_gal_html += 'var selV=document.getElementById("gal_var"), selT=document.getElementById("gal_typ");'
_gal_html += 'function upd(){'
_gal_html += 'var cells=document.querySelectorAll(".gal_cell");'
_gal_html += 'for(var i=0;i<cells.length;i++){'
_gal_html += 'var c=cells[i];'
_gal_html += 'c.style.display=(c.dataset.var===selV.value && c.dataset.typ===selT.value)?"":"none";'
_gal_html += '}}'
_gal_html += 'selV.addEventListener("change",upd); selT.addEventListener("change",upd); upd();'
_gal_html += '})();'
_gal_html += '</script></div>'
display(HTML(_gal_html))